<a href="https://colab.research.google.com/github/MevinIIT/CM2603-DSGP-Group-3/blob/Mevin/DSGP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ---------------------------------------------------------
# 1. LOAD AND PREPARE DATA
# ---------------------------------------------------------
# Load the datasets
files = {
    'Abnormal': "ECG Images of Patient that have abnormal heartbeat (233x12=2796)_flattened.csv",
    'History_MI': "ECG Images of Patient that have History of MI (172x12=2064)_flattened.csv",
    'Normal': "Normal Person ECG Images (284x12=3408)_flattened.csv",
    'MI': "ECG Images of Myocardial Infarction Patients (240x12=2880)_flattened.csv"
}

dfs = []
for label, filepath in files.items():
    df = pd.read_csv(filepath)
    # The 'Condition' is our target variable
    if label == 'Abnormal': df['Target'] = 'Abnormal Heartbeat'
    elif label == 'History_MI': df['Target'] = 'History of MI'
    elif label == 'Normal': df['Target'] = 'Normal'
    elif label == 'MI': df['Target'] = 'Myocardial Infarction'
    dfs.append(df)

# Combine all data
full_data = pd.concat(dfs, ignore_index=True)

# Separate Features (ECG signals) and Target (Condition)
X = full_data.drop(columns=['filename', 'Target']).fillna(0)
y = full_data['Target']

# Encode labels (Normal, MI, etc.) into numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 2. TRAIN THE PREDICTION MODEL
# ---------------------------------------------------------
print("Training Risk Model... (this may take a moment)")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print(f"Model Trained. Accuracy: {rf_model.score(X_test, y_test):.2%}")

# ---------------------------------------------------------
# 3. DEFINE THE RISK CALCULATOR (ECG + HABITS)
# ---------------------------------------------------------
def calculate_total_risk(ecg_data, habits):
    """
    Calculates final risk percentage based on ECG Model + Habits.
    """
    # Step A: Get Baseline Risk from ECG Model
    # We predict the probability of the class 'Normal'.
    # Base Risk = Probability of ANY condition (1 - Probability of Normal)
    probs = rf_model.predict_proba(ecg_data.reshape(1, -1))[0]
    classes = rf_model.classes_

    # Find index of 'Normal' class
    normal_idx = np.where(le.classes_ == 'Normal')[0][0]
    prob_normal = probs[normal_idx]

    # Base Risk Score (0 to 100)
    base_risk = (1 - prob_normal) * 100

    # Step B: Adjust Risk based on Habits
    # Multipliers: >1 increases risk, <1 decreases risk
    multiplier = 1.0

    if habits.get('smoker'):
        multiplier *= 1.40  # Smoking increases risk by 40%
    if habits.get('sedentary'):
        multiplier *= 1.25  # Lack of exercise increases risk by 25%
    if habits.get('high_alcohol'):
        multiplier *= 1.20  # Alcohol increases risk by 20%

    if habits.get('exercise'):
        multiplier *= 0.80  # Regular exercise decreases risk by 20%
    if habits.get('healthy_diet'):
        multiplier *= 0.85  # Good diet decreases risk by 15%

    final_risk = base_risk * multiplier

    # Cap risk at 99%
    final_risk = min(final_risk, 99.0)

    return base_risk, final_risk, le.classes_[np.argmax(probs)]

# ---------------------------------------------------------
# 4. RUN A TEST CASE
# ---------------------------------------------------------
# Let's pick a random patient from the test set
sample_idx = 0
sample_ecg = X_test.iloc[sample_idx].values

# Scenario 1: Patient has BAD habits
habits_bad = {'smoker': True, 'sedentary': True, 'exercise': False}
base, final, pred = calculate_total_risk(sample_ecg, habits_bad)

print(f"\n--- Patient Analysis (Predicted Condition: {pred}) ---")
print(f"Baseline Physiological Risk (ECG only): {base:.2f}%")
print(f"Risk with Bad Habits (Smoking, Sedentary): {final:.2f}% (RISK INCREASED)")

# Scenario 2: Same Patient has GOOD habits
habits_good = {'smoker': False, 'sedentary': False, 'exercise': True, 'healthy_diet': True}
base, final, pred = calculate_total_risk(sample_ecg, habits_good)

print(f"Risk with Good Habits (Exercise, Diet):    {final:.2f}% (RISK DECREASED)")

FileNotFoundError: [Errno 2] No such file or directory: 'ECG Images of Patient that have abnormal heartbeat (233x12=2796)_flattened.csv'

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

def run_risk_prediction_xgboost():
    print("--- SYSTEM STARTUP (XGBOOST VERSION) ---")

    # ======================================================
    # 1. LOAD TRAINING DATA
    # ======================================================
    files = {
        'Abnormal': "ECG Images of Patient that have abnormal heartbeat (233x12=2796)_flattened.csv",
        'History_MI': "ECG Images of Patient that have History of MI (172x12=2064)_flattened.csv",
        'Normal': "Normal Person ECG Images (284x12=3408)_flattened.csv",
        'MI': "ECG Images of Myocardial Infarction Patients (240x12=2880)_flattened.csv"
    }

    dfs = []
    print("1. Loading Training Datasets...")
    for label, filepath in files.items():
        try:
            df = pd.read_csv(filepath)
            if label == 'Abnormal': df['Target'] = 'Abnormal Heartbeat'
            elif label == 'History_MI': df['Target'] = 'History of MI'
            elif label == 'Normal': df['Target'] = 'Normal'
            elif label == 'MI': df['Target'] = 'Myocardial Infarction'
            dfs.append(df)
        except FileNotFoundError:
            print(f"Error: File {filepath} not found.")
            return

    # Combine Data
    full_data = pd.concat(dfs, ignore_index=True)
    X = full_data.drop(columns=['filename', 'Target']).fillna(0)
    y = full_data['Target']

    # Encode Labels (XGBoost requires numeric target labels 0, 1, 2...)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # ======================================================
    # 2. TRAIN XGBOOST MODEL
    # ======================================================
    print("2. Training XGBoost Model... (This is usually faster)")

    # Initialize XGBoost Classifier
    model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42
    )

    model.fit(X, y_encoded)
    print("   Model Trained Successfully.")

    # ======================================================
    # 3. PREDICT ON PATIENT
    # ======================================================
    input_ecg_file = "patient_test_input.csv"
    input_lifestyle_file = "patient_lifestyle.csv"

    print("\n3. Reading Patient Data...")
    try:
        # Load Patient ECG
        patient_ecg = pd.read_csv(input_ecg_file)
        if 'filename' in patient_ecg.columns:
            patient_ecg = patient_ecg.drop(columns=['filename'])

        # Load Lifestyle
        lifestyle_df = pd.read_csv(input_lifestyle_file)
        habits = lifestyle_df.iloc[0].to_dict()

    except FileNotFoundError as e:
        print(f"Error: {e}")
        return

    # ======================================================
    # 4. CALCULATE RISK
    # ======================================================
    # Get Probabilities from XGBoost
    probs = model.predict_proba(patient_ecg)[0]

    # Calculate Base Risk (1 - Probability of Normal)
    normal_index = list(le.classes_).index('Normal')
    prob_normal = probs[normal_index]
    base_risk = (1.0 - prob_normal) * 100

    # Get Diagnosis
    pred_idx = np.argmax(probs)
    diagnosis = le.classes_[pred_idx]

    # Apply Lifestyle Multipliers
    multiplier = 1.0
    if habits.get('smoker'):       multiplier += 0.40
    if habits.get('high_alcohol'): multiplier += 0.20
    if habits.get('sedentary'):    multiplier += 0.25
    if habits.get('exercise'):     multiplier -= 0.20
    if habits.get('healthy_diet'): multiplier -= 0.15

    multiplier = max(0.1, multiplier)
    final_risk = min(base_risk * multiplier, 99.9)

    # ======================================================
    # 5. REPORT
    # ======================================================
    print("\n" + "="*50)
    print("      XGBOOST RISK ASSESSMENT REPORT")
    print("="*50)
    print(f"AI DIAGNOSIS:          {diagnosis}")
    print(f"Physiological Risk:    {base_risk:.2f}%")
    print("-" * 50)
    print(f"Lifestyle Adjustment:  x{multiplier:.2f}")
    print(f"FINAL RISK PERCENTAGE: {final_risk:.2f}%")
    print("="*50)

if __name__ == "__main__":
    run_risk_prediction_xgboost()